<a href="https://colab.research.google.com/github/devesssi/llm-diffusion-models-finetuning/blob/main/finetuning(Astrology).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from datasets import load_dataset

dataset = load_dataset("karthiksagarn/astro_horoscope", split = "train")


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

horoscope.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21959 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['sign', 'category', 'date', 'horoscope'],
    num_rows: 21959
})

In [ ]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

model = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model)

def tokenize(batch):
  return tokenizer( batch["horoscope"] , truncation = True , max_length= 512, )

dataset = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Map:   0%|          | 0/21959 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer , mlm = False)

### Masked Language Modeling (MLM) vs. Causal Language Modeling (CLM)

These are two primary pre-training objectives used for Transformer-based language models, influencing how a model understands and generates text.

#### Masked Language Modeling (MLM)

*   **How it works**: In MLM, a percentage of the tokens in the input sequence are randomly masked (replaced with a special `[MASK]` token, a random token, or kept unchanged). The model's task is to predict the original masked tokens based on their context (both left and right context).
*   **Characteristics**: This objective allows the model to learn a *bidirectional* understanding of language, meaning it can consider words both before and after a given word to understand its meaning.
*   **Typical Models**: Models like **BERT (Bidirectional Encoder Representations from Transformers)** are pre-trained using MLM. They excel at tasks requiring deep contextual understanding, such as sentiment analysis, question answering, and named entity recognition.
*   **Tokenizer Requirement**: Requires a tokenizer that includes a `mask_token` in its vocabulary.

#### Causal Language Modeling (CLM)

*   **How it works**: In CLM, the model is trained to predict the *next* token in a sequence, given all the preceding tokens. It processes the text from left to right, only looking at the past context.
*   **Characteristics**: This objective leads to models that are particularly good at *generating* text sequentially, as they learn the statistical dependencies for continuation. It's often referred to as 'autoregressive' modeling.
*   **Typical Models**: Models like **GPT (Generative Pre-trained Transformer)** and **Qwen** are pre-trained using CLM. They are widely used for text generation, summarization, translation, and chatbots.
*   **Tokenizer Requirement**: Does *not* require a `mask_token` because there's no masking involved; the task is purely about predicting the next token.

***

The `Qwen/Qwen3-0.6B` model, which you are using, is a causal language model. This means its tokenizer does not have a `mask_token`, and therefore it cannot be used for Masked Language Modeling (`mlm=True`). Instead, it's designed for Causal Language Modeling, where `mlm=False` is the appropriate setting for the data collator to prepare sequences for next-token prediction.

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

import torch

model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")
model = torch.compile(model)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
training_args = TrainingArguments(
    output_dir="qwen3-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,  # Increased from 2 to 4
    gradient_accumulation_steps=8,
    gradient_checkpointing=False, # Keeping as False for speed unless OOM
    bf16=True,  # Changed to True for speedup on compatible GPUs
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    remove_unused_columns=False, # Add this line to fix the ValueError
)

This code block sets up and initiates the fine-tuning process for your language model using the `Trainer` class from the Hugging Face Transformers library.

Let's go through each part:

1.  **`trainer = Trainer(...)`**:
    This line initializes the `Trainer` object, which is a high-level API designed to make fine-tuning models from the Transformers library straightforward.
    *   `model=model`: This passes the `AutoModelForCausalLM` you loaded earlier to the trainer. This is the model that will be fine-tuned.
    *   `args=training_args`: This is where your `TrainingArguments` object comes into play. It contains all the hyperparameters and configurations for training, such as the number of epochs, batch size, learning rate, output directory, and evaluation strategy.
    *   `train_dataset=dataset["train"]`: This specifies the dataset to be used for training the model. It's the 'train' split of your processed dataset.
    *   `eval_dataset=dataset["test"]`: This specifies the dataset to be used for evaluating the model's performance during training. It's the 'test' split of your processed dataset.
    *   `processing_class=tokenizer`: While `tokenizer` is passed here, it's more common to use `tokenizer` for `Trainer` or just let the `data_collator` handle tokenization if `raw_datasets` are passed. However, it effectively ensures the tokenizer is available for any internal processing the `Trainer` might need.
    *   `data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)`: This object is responsible for preparing batches of data for the model. It takes your tokenized inputs and formats them appropriately. Since `mlm=False`, it's set up for Causal Language Modeling (CLM), which means it will shift the labels for next-token prediction, as discussed earlier.

2.  **`trainer.train()`**:
    This is the core command that starts the model training process. The `Trainer` will use the `model`, `training_args`, `train_dataset`, `eval_dataset`, and `data_collator` to:
    *   Iterate over the training data for the specified number of epochs.
    *   Perform forward and backward passes.
    *   Update model weights using the optimizer and learning rate scheduler defined in `training_args`.
    *   Log metrics and save checkpoints according to your `logging_steps` and `save_strategy`.
    *   Evaluate the model on the `eval_dataset` at specified intervals (`eval_strategy`).

3.  **`trainer.push_to_hub()`**:
    After training is complete, this command uploads your fine-tuned model and its tokenizer to the Hugging Face Hub. This makes your model easily shareable, reproducible, and accessible for others (or yourself) to download and use later. It will require you to be logged in to Hugging Face.

Let's go through each of the `TrainingArguments`:

*   `output_dir`: This specifies the directory where the model checkpoints and other training outputs (like logs and predictions) will be saved. In your case, it's set to `"qwen3-finetuned"`.

*   `num_train_epochs`: This is the total number of times the model will iterate over the entire training dataset. You've set it to `3`, meaning the model will see each training example three times.

*   `per_device_train_batch_size`: This defines the batch size per GPU/TPU core during training. You've set it to `2`. This means that on each device, `2` samples will be processed at a time.

*   `gradient_accumulation_steps`: When this is greater than 1, gradients are accumulated over multiple batches before performing a single optimization step. This effectively allows you to use a larger 'effective' batch size than `per_device_train_batch_size` would suggest, without requiring more GPU memory. With `per_device_train_batch_size=2` and `gradient_accumulation_steps=8`, your effective batch size is `2 * 8 = 16`.

*   `gradient_checkpointing`: If `True`, this technique saves memory by trading off computation. It recomputes activations during the backward pass instead of storing them, which can be useful for training very large models that might otherwise run out of memory.

*   `bf16`: If `True`, it enables training in bfloat16 precision. This can speed up training and reduce memory usage on hardware that supports it (like NVIDIA Ampere GPUs and newer, or TPUs), while typically maintaining numerical stability better than float16 for deep learning models.

*   `learning_rate`: This is the initial learning rate for the optimizer. It controls how large a step the optimizer takes in the direction of the gradient during each update. You've set it to `2e-5` (0.00002).

*   `logging_steps`: This specifies how often (in steps) the training metrics (like loss) are logged to the console and to any integrated logging tools (e.g., TensorBoard, Weights & Biases). You've set it to `10`, so logs will appear every 10 steps.

*   `eval_strategy`: This determines when evaluation is performed. Setting it to `"epoch"` means the model will be evaluated on the validation set at the end of each training epoch.

*   `save_strategy`: Similar to `eval_strategy`, this dictates when the model checkpoints are saved. `"epoch"` means a checkpoint will be saved after each epoch.

*   `load_best_model_at_end`: If `True`, after training is complete, the `Trainer` will load the best model checkpoint found during training (based on the evaluation metric) back into the model. This ensures you end up with the best performing model, not necessarily the last one.

Gradient accumulation steps let you train a model like you're using a big batch of data, even if your GPU can only handle small batches.

Here’s how it works — very simply:

You run a small batch through the model and calculate the gradients (how to improve).
Instead of updating the model right away, you save those gradients.
You do the same for a few more small batches, adding up all the gradients.
Only after several steps (e.g., 4 or 8) do you update the model using the combined gradients.
👉 This acts like you used one big batch, which helps training be more stable — but without needing more GPU memory.

Example:
You want a batch size of 64, but your GPU only fits 16.
Use 4 gradient accumulation steps → process 4 batches of 16 → update once.
= Same effect as a batch of 64.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()
trainer.push_to_hub()

W0323 09:48:53.587000 2740 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/usr/local/lib/python3.12/dist-

Epoch,Training Loss,Validation Loss
